# `numpy`支持的线性代数运算

In [1]:
#以别名形式导入numpy. 
import numpy as np; 
#导入与ndarray有关的数据类型. 
from numpy import int8, int16, int32, int64; 
from numpy import uint8, uint16, uint32, uint64; 
from numpy import float16, float32, float64; 
from numpy import complex64, complex128; 

In [2]:
#从numpy中单独导入子模块linalg. 
from numpy import linalg as npla; 

## 线性运算
同维向量, 同型矩阵的加法, 数乘

In [3]:
vctA = np.array([1, 0], dtype = float64); 
vctB = np.array([-1, -1], dtype = float64); 

In [4]:
print(vctA + vctB, -2 * vctB); 

[ 0. -1.] [2. 2.]


## 乘法运算

### 向量的数量积
用法: 
```python
v1.dot(v2)
```
其他等效形式
```python
np.dot(v1, v2)
```
`v1`, `v2` 参与数量积运算的向量, 要求
* 均为`np.ndarray`对象
* `ndim`均为`1`, `shape`属性值相同. 


In [5]:
print(np.dot(vctA, vctB)); 

-1.0


### 矩阵的乘法
用法: 
```python
A.dot(B)
```
其他等效形式
```python
np.dot(A, B)
```

`A`, `B` 参与乘法的矩阵, 要求
* 均为`np.ndarray`对象
* `ndim`均为`2`, `A.shape[1] == B.shape[0]`

多矩阵连乘可采用`np.linalg.multidot`方法

In [6]:
#定义二维旋转变换矩阵(绕原点旋转)
def rot2D_matrix(rot_angle): 
    cθ = np.cos(rot_angle); sθ = np.sin(rot_angle); 
    ls = [[cθ, -sθ], [sθ, cθ]]; 
    return(np.array(ls, dtype = float64)); 

In [7]:
matA = rot2D_matrix(np.pi / 5); 
matB = rot2D_matrix(- np.pi / 4); 
print(vctA); print(matB.dot(vctA)); 
print(np.dot(matB, matB)); 

[1. 0.]
[ 0.70710678 -0.70710678]
[[ 0.  1.]
 [-1.  0.]]


### 矩阵乘法的逆
* 对于非奇异方阵`A`, 使用`np.linalg.inv`求逆. 
    ```python
    np.linalg.inv(A)
    ```
    
* 对于行列数不相等的矩阵, 或者奇异方阵`S`, 使用`np.linalg.pinv`求Moore-Penrose广义逆. 
    ```python
    np.linalg.pinv(S)
    ```
    
    使用`np.linalg.inv`求逆将报错`LinAlgError`

In [8]:
matInvA = npla.inv(matA); 
print(matInvA, rot2D_matrix(- np.pi / 5), sep = "\n\n")

[[ 0.80901699  0.58778525]
 [-0.58778525  0.80901699]]

[[ 0.80901699  0.58778525]
 [-0.58778525  0.80901699]]


### 方阵的幂
用法: 
```python
np.linalg.matrix_power(A, k)
```
* `A` 参与幂运算的方阵, `np.ndarray`对象, 要求`ndim`为`2`, `shape[0] == shape[1]`; 
* `k` 幂指数, `int`对象

In [9]:
#由于numpy.pi不是一个精确值, 因此计算结果在数值上与(- numpy.identity(2))之间存在一定差异. 
print(npla.matrix_power(matInvA, 5)) 

[[-1.00000000e+00 -5.55111512e-17]
 [ 1.11022302e-16 -1.00000000e+00]]


## 初等变换

### 三种初等变换

|初等变换|语句|
|:-:|:-|
|$$r_i \times k$$ $$c_i \times k$$|`A[i, : ] *= k`<br>`A[: , i] *= k`|
|$$r_i + k \cdot r_j$$ $$c_i + k \cdot c_j$$|`A[i, : ] += k * A[j, : ]`<br>`A[: , i] += k * A[: , j]`|
|$$r_i \leftrightarrow r_j$$ $$c_i \leftrightarrow c_j$$|`A[(i, j), : ] = A[(j, i), : ]`<br>`A[: , (i, j)] = A[: , (j, i)]`|

### 行列式与秩
* 对于方阵`A`, 使用`np.linalg.det`求行列式. 
    ```python
    np.linalg.det(A)
    ```
    对于行列数不相等的矩阵, 使用`np.linalg.det`求行列式将报错`LinAlgError`

* 对于任意形状的矩阵`A`, 使用`np.linalg.matrix_rank`求秩. 
    ```python
    np.linalg.matrix_rank(A)
    ```

In [10]:
#旋转变换是正交变换, 因此行列式为1. 
#由于numpy.pi不是一个精确值, 因此计算结果在数值上与1之间存在一定差异. 
print(npla.det(matA), npla.det(matB))

1.0000000000000002 1.0


In [11]:
#旋转变换是可逆的, 因此满秩. 
print(npla.matrix_rank(matA)); 
print(npla.matrix_rank(matA) == max(matA.shape))

2
True


## 线性方程组

### 线性方程组的特解
* 当关于$\mathbf{x}$的线性方程组
    $$\mathbf{A} \mathbf{x} = \mathbf{b} ~ (\mathbf{A} \in \mathcal{M(n, \mathbb{R})}, \mathbf{x}, \mathbf{b} \in \mathbb{R}^n)$$
    有唯一解时, 使用`np.linalg.solve`求唯一解
    ```python
    np.linalg.solve(A, b)
    ```
    对于欠定或超定方程组, 使用`np.linalg.solve`求特解将报错`LinAlgError`
* 对于**齐次线性方程组**, 无论该方程组是否存在非零解, `np.linalg.solve`方法**总是返回其零解**. 

在**`numpy 1.16`中, 尚未提供**计算齐次线性方程组基础解系的`np.linalg.null_space`方法**

In [12]:
print(npla.solve(matB, vctA))

[0.70710678 0.70710678]


### 线性方程组的最小二乘解
* 当关于$\mathbf{x}$的线性方程组
    $$\mathbf{A} \mathbf{x} = \mathbf{b}$$
    处于欠定或超定状态时, 使用`np.linalg.lstsq`通过奇异值分解法求线性最小二乘解
    
    ```python
    result = np.linalg.lstsq(A, b, rcond=-1)
    ```

`np.linalg.lstsq`的返回值`result`为四元`tuple`对象, 其中: 
* `result[0]`为最小二乘解向量$\mathbf{x}_0$, 使得
$$\forall \mathbf{x} \in \mathbb{R}^n, \left | \mathbf{A} \mathbf{x}_0 - \mathbf{b} \right | \le \left | \mathbf{A} \mathbf{x} - \mathbf{b} \right |$$
* `result[1]`为残差平方和
$${\left | \mathbf{A} \mathbf{x}_0 - \mathbf{b} \right |}^2$$
* `result[2]`为系数矩阵$\mathbf{A}$的秩
* `result[3]`为系数矩阵$\mathbf{A}$的奇异值

In [13]:
print(npla.lstsq(np.array([[1], [1], [1]]), 
    np.array([0, 1, 2]), rcond=-1)
 )

(array([1.]), array([2.]), 1, array([1.73205081]))


## 特征值, 特征向量与矩阵分解

### 特征值与特征向量的计算
* 对于$n ~ (n \ge 2)$阶方阵`A`, 使用`np.linalg.eig`求特征值和对应的特征向量
    ```python
    eigsys = np.linalg.eig(A)
    ```
    对于1阶方阵, 或者行列数不相等的矩阵, 使用`np.linalg.eig`求特征值和特征向量将报错`LinAlgError`

`np.linalg.eig`的返回值`eigsys`为二元`tuple`对象, 其中: 
* `eigsys[0]`为矩阵`A`的$n$个特征值(包括特征多项式的重根)
    $$(\lambda_1, \dots, \lambda_n)$$
* `eigsys[1]`为矩阵`A`的与$n$个特征值一一对应的特征向量构成的**列向量组**
    $$\mathbf{P} = (\mathbf{p}_1, \dots, \mathbf{p}_n)$$
    
**警告: 不要试图使用特征向量来计算齐次线性方程组基础解系**
> 求特征多项式 $\left | \mathbf{A} - \lambda \mathbf{E} \right | $ 零点以解算特征值$\lambda_i$的数值计算过程中, 迭代计算所引入和放大的**舍入误差, 会导致特征值被误判**, 从而**遗漏**或者**错算**部分基向量

In [14]:
λ, P = npla.eig(rot2D_matrix(np.pi / 3))
print(λ, P, sep = "\n\n")

[0.5+0.8660254j 0.5-0.8660254j]

[[0.70710678+0.j         0.70710678-0.j        ]
 [0.        -0.70710678j 0.        +0.70710678j]]


### 矩阵的分解

|分解方法|`numpy.linalg`方法名称|备注|
|:-:|:-|:-|
|$Q R$分解|`np.linalg.qr(A)`|结果为二元`tuple`, 其中元素顺次为分解后<br>所得的正交矩阵$\mathbf{Q}$和上三角矩阵$\mathbf{R}$, 满足<br>$\mathbf{A} = \mathbf{Q} \mathbf{R}$|
|SVD分解|`np.linalg.svd(A)`|结果为三元`tuple`, 其中元素顺次为分解后<br>所得的左正交矩阵$\mathbf{U}$, 奇异值矩阵$\mathbf{S}$和右正交<br>矩阵$\mathbf{V}$, 满足$\mathbf{A} = \mathbf{U} \mathbf{S} \mathbf{V}^\mathrm{H}$|
|Cholesky分解|`np.linalg.cholesky(A)`|矩阵`A`必须是正定的, 否则报错`LinAlgError`; <br>结果为为分解后所得的下三角矩阵$\mathbf{L}$, 满足<br>$\mathbf{A} = \mathbf{L} \mathbf{L}^\mathrm{H}$|